In [ ]:
from datasets import load_dataset, load_from_disk
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt
from functools import reduce
import hashlib
from typing import List, Dict, Tuple
import ast
import pandas as pd
import numpy as np
import json

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

SAVE_DATASET_PATH = "../../data/qa_datasets/trivia_qa_rcwikipedia_validation/original"
SAVE_CONTEXTS_PATH = "../../data/qa_datasets/trivia_qa_rcwikipedia_validation/relevant_contexts.csv"
SAVE_TMP_CONTEXTS_PATH = "../../data/qa_datasets/trivia_qa_rcwikipedia_validation/tmp_triviaqa_cntx.csv"
SAVE_QA_PATH = "../../data/qa_datasets/trivia_qa_rcwikipedia_validation/qa_pairs.csv"
SAVE_STATS_PATH = "../../data/qa_datasets/trivia_qa_rcwikipedia_validation/stats.json"
CNTX_MAX_LEN = 1024
CNTX_MIN_LEN = 64
MAX_QA_PAIRS = 500

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CNTX_MAX_LEN,
    separators=["\n\n"],
    chunk_overlap=64,
    length_function=len,
    is_separator_regex=False,
)

# text_splitter = CharacterTextSplitter(
#     separator="\n\n",
#     length_function=len,
#     is_separator_regex=False,
# )

In [ ]:
!pip install langchain_text_splitters

#### Скачиваем датасет с Hugging Face

In [ ]:
# https://huggingface.co/datasets/mandarjoshi/trivia_qa
valid_dataset = load_dataset("mandarjoshi/trivia_qa", "rc.wikipedia", split='validation') # 7990
valid_dataset.save_to_disk(SAVE_DATASET_PATH)

In [ ]:
valid_dataset = load_from_disk(SAVE_DATASET_PATH)

In [ ]:
valid_dataset

#### Извлекаем все уникальные контексты (разбиваем их на чанки, т.к. они достаточно длинные)

In [ ]:
base_contexts = list(map(lambda v: (v['title'], v['wiki_context']), valid_dataset['entity_pages']))

In [ ]:
unique_splitted_contexts = dict()
contexts_qa_idxes = defaultdict(list)

process = tqdm(range(valid_dataset.num_rows))
scntx_counter = 0
for i in process:
    contexts = base_contexts[i]

    process.set_postfix_str(f"unique contexts: {len(unique_splitted_contexts)} / {scntx_counter}")

    for j,title in enumerate(contexts[0]): # titles
        if len(title) < 1:
            raise ValueError
        
        base_context = contexts[1][j] # context
        splitted_context = text_splitter.create_documents([base_context])

        for scntx in splitted_context:
            scntx_hash = hashlib.sha1(scntx.page_content.encode()).hexdigest()
        
            unique_splitted_contexts[scntx_hash] = (title, scntx.page_content)
            contexts_qa_idxes[scntx_hash].append(i)
            scntx_counter += 1

In [ ]:
cntx_df_dict = {'context': [], 'title': [],'qa_idxes': []}
for cntx_hash in contexts_qa_idxes.keys():
    cntx_df_dict['title'].append(unique_splitted_contexts[cntx_hash][0])
    cntx_df_dict['context'].append(unique_splitted_contexts[cntx_hash][1])
    cntx_df_dict['qa_idxes'].append(contexts_qa_idxes[cntx_hash])

In [ ]:
cntx_lens = list(map(lambda cntx: len(cntx), cntx_df_dict['context']))
plt.hist(cntx_lens, bins=50, density=True)
plt.title("Context len distribution")
plt.grid()
plt.show()

In [ ]:
cntx_df = pd.DataFrame(cntx_df_dict)
cntx_df.to_csv(SAVE_TMP_CONTEXTS_PATH, index=False)

#### Фильтруем контексты по длине и максимальному количеству qa-пар

In [ ]:
cntx_df = pd.read_csv(SAVE_TMP_CONTEXTS_PATH)
cntx_df['qa_idxes'] = cntx_df['qa_idxes'].map(lambda v: ast.literal_eval(v))

In [ ]:
print(f"before: {cntx_df.shape}")

wolcnt_cntx_df = cntx_df[cntx_df['context'].str.len() <= CNTX_MAX_LEN].reset_index(drop=True)
print(f"after long contexts deletion: {wolcnt_cntx_df.shape}")

qa_idx_to_delete = set(reduce(lambda acc, v: acc + v, cntx_df[cntx_df['context'].str.len() > CNTX_MAX_LEN]['qa_idxes'].tolist(), []))
for i in tqdm(range(wolcnt_cntx_df.shape[0])):
    wolcnt_cntx_df.loc[i, 'qa_idxes'] =  list(set(wolcnt_cntx_df['qa_idxes'][i]).difference(qa_idx_to_delete))
wolcnt_cntx_df = wolcnt_cntx_df[wolcnt_cntx_df['qa_idxes'].str.len() > 0].reset_index(drop=True)
print(f"after incomplete contexts deletion: {wolcnt_cntx_df.shape}")

wolscnt_cntx_df = wolcnt_cntx_df[wolcnt_cntx_df['context'].str.len() >= CNTX_MIN_LEN].reset_index(drop=True)
print(f"after small contexts deletion: {wolcnt_cntx_df.shape}")

qa_idx_to_delete = set(reduce(lambda acc, v: acc + v, wolcnt_cntx_df[wolcnt_cntx_df['context'].str.len() < CNTX_MIN_LEN]['qa_idxes'].tolist(), []))
for i in tqdm(range(wolscnt_cntx_df.shape[0])):
    wolscnt_cntx_df.loc[i, 'qa_idxes'] =  list(set(wolscnt_cntx_df['qa_idxes'][i]).difference(qa_idx_to_delete))
wolscnt_cntx_df = wolscnt_cntx_df[wolscnt_cntx_df['qa_idxes'].str.len() > 0].reset_index(drop=True)
print(f"after incomplete contexts deletion: {wolscnt_cntx_df.shape}")

unique_qa_idxes = sorted(list(set(reduce(lambda acc, v: acc + v, wolscnt_cntx_df['qa_idxes'].tolist(), []))))[:MAX_QA_PAIRS]
for i in range(wolscnt_cntx_df.shape[0]):
    wolscnt_cntx_df.loc[i, 'qa_idxes'] =  list(set(wolscnt_cntx_df['qa_idxes'][i]).intersection(unique_qa_idxes))
filtered_cntx_df = wolscnt_cntx_df[wolscnt_cntx_df['qa_idxes'].str.len() > 0].reset_index(drop=True)
print(f"after ommiting qa-pairs: {filtered_cntx_df.shape}")

In [ ]:
cntx_lens = list(map(lambda cntx: len(cntx), filtered_cntx_df['context']))
plt.hist(cntx_lens, bins=50, density=True)
plt.title("Context len distribution")
plt.grid()
plt.show()

In [ ]:
filtered_cntx_df.to_csv(SAVE_CONTEXTS_PATH, index=False)

#### Формируем отдельный датасет с qa-парами (с учётом имеющегося набора отфильтрованных контекстов)

In [ ]:
qa_df_dict = {'answer': [], 'question': [], 'qa_idx': []}

all_questions = valid_dataset['question']
all_answers = valid_dataset['answer']

for qa_idx in tqdm(unique_qa_idxes):
    qa_df_dict['answer'].append(all_answers[qa_idx]['value'])
    qa_df_dict['question'].append(all_questions[qa_idx])
    qa_df_dict['qa_idx'].append(qa_idx)

In [ ]:
answer_lens = list(map(lambda cntx: len(cntx), qa_df_dict['answer']))
plt.hist(answer_lens, bins=50)
plt.title("Answer len distribution")
plt.grid()
plt.show()

In [ ]:
question_lens = list(map(lambda cntx: len(cntx), qa_df_dict['question']))
plt.hist(question_lens, bins=50)
plt.title("Question len distribution")
plt.grid()
plt.show()

In [ ]:
qa_df = pd.DataFrame(qa_df_dict)
print(qa_df.shape)

In [ ]:
print("Количество уникальных вопросов: ", len(qa_df['question'].unique()))

In [ ]:
qa_df.to_csv(SAVE_QA_PATH, index=False)

#### Сохраняем статистику по полученным csv-файлам с контекстами и соответствующими qa-парами

In [ ]:
qa_df = pd.read_csv(SAVE_QA_PATH)
filtered_cntx_df = pd.read_csv(SAVE_CONTEXTS_PATH)

In [ ]:
def compute_stats(numbers: List[int]) -> Dict[str, float]:
    return {'median': round(np.median(numbers),5),
            'mean': round(np.mean(numbers),5),
            'std': round(np.std(numbers),5),
            'min': round(min(numbers),5),
            'max': round(max(numbers),5)}

In [ ]:
filtered_ds_statistics = {
    'contexts_amount': filtered_cntx_df.shape[0],
    'contexts_len': compute_stats(list(map(lambda v: len(v), filtered_cntx_df['context'].tolist()))),
    'qa_pairs_amount': qa_df.shape[0],
    'answers_len':  compute_stats(list(map(lambda v: len(v), qa_df['answer'].tolist()))),
    'questions_len': compute_stats(list(map(lambda v: len(v), qa_df['question'].tolist())))
}

In [ ]:
with open(SAVE_STATS_PATH, 'w', encoding='utf-8') as fd:
    json.dump(filtered_ds_statistics, fd, indent=1, ensure_ascii=False)